In [0]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime

BLS_URL = "https://download.bls.gov/pub/time.series/pr/"

DATAUSA_URL = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

BLS_DIR = "/Volumes/rearc/bls/raw/"
DATAUSA_DIR = "/Volumes/rearc/datausa/raw/"

os.makedirs(BLS_DIR, exist_ok=True)
os.makedirs(DATAUSA_DIR, exist_ok=True)

headers = {
    "User-Agent": "idontwantthisingithub@someemaildomain.com"
}

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# -----------------------------
# BLS
# -----------------------------

response = requests.get(BLS_URL, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

source_files = set()

for link in soup.find_all("a", href=True):
    href = link["href"]

    # Ignore directories
    if href.endswith("/"):
        continue

    filename = os.path.basename(href)

    if not filename:
        continue

    source_files.add(filename)

    file_url = urljoin(BLS_URL, href)
    file_path = os.path.join(BLS_DIR, f"{filename}.{timestamp}")

    r = requests.get(file_url, headers=headers)
    r.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(r.content)

    print("DOWNLOADED:", f"{filename}.{timestamp}")


# Remove files no longer at the BLS source
for filename in os.listdir(BLS_DIR):
    original_filename = filename.split('.')[0]
    if original_filename not in source_files:
        os.remove(os.path.join(BLS_DIR, filename))
        print("REMOVED:", filename)


# -----------------------------
# Data USA
# -----------------------------

response = requests.get(DATAUSA_URL)
response.raise_for_status()

population_file = f"{DATAUSA_DIR}/population.{timestamp}.json"
with open(population_file, "wb") as f:
    f.write(response.content)

print("DOWNLOADED:", f"population.{timestamp}.json")

---------------------------------------------------------------------------
gaierror                                  Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/urllib3/connection.py:198, in HTTPConnection._new_conn(self)
    197 try:
--> 198     sock = connection.create_connection(
    199         (self._dns_host, self.port),
    200         self.timeout,
    201         source_address=self.source_address,
    202         socket_options=self.socket_options,
    203     )
    204 except socket.gaierror as e:

File /databricks/python/lib/python3.12/site-packages/urllib3/util/connection.py:60, in create_connection(address, timeout, source_address, socket_options)
     58     raise LocationParseError(f"'{host}', label empty or too long") from None
---> 60 for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
     61     af, socktype, proto, canonname, sa = res

File /usr/lib/python3.12/socket.py:963, in getaddrinfo(host, port, f

In [0]:
import socket
for host in ("pypi.org", "www.google.com", "download.bls.gov", "honolulu-api.datausa.io"):
    try:
        print(f"{host:<28} OK   {socket.gethostbyname(host)}")
    except Exception as e:
        print(f"{host:<28} FAIL {e}")

pypi.org                     OK   151.101.128.223
www.google.com               FAIL [Errno -2] Name or service not known
download.bls.gov             FAIL [Errno -2] Name or service not known
honolulu-api.datausa.io      FAIL [Errno -2] Name or service not known


In [0]:
display(dbutils.fs.ls(VOL))

path,name,size,modificationTime
dbfs:/Volumes/rearc/data-quest/step1_folder/bls/,bls/,0,1786673976920
dbfs:/Volumes/rearc/data-quest/step1_folder/datausa/,datausa/,0,1786673976920
